# HW-3: Hash table

NOTE: I have taken the help of Claude AI 

## Part I: Hash table implementation

### INIT method

    def __init__(self, size=101):
        self.size = size
        self.count = 0
        # Array of buckets — each bucket is a list of [key, value] pairs (chaining)
        self.table = [[] for _ in range(self.size)]

Used array method as was suggested in the instructions.

## Search method

    def search(self,key):
        index = self._hash(key, method="division")
        bucket = self.table[index]

        for pair in bucket:
            if pair[0] == key:
                return pair[1]

        raise KeyError(f"Key '{key}' not found")

Will raise an error if the key is not stored

## INSERT method 


    def insert(self,key,value):
        index = self._hash(key, method="division")
        bucket = self.table[index]

        # If key already exists, update the value
        for pair in bucket:
            if pair[0] == key:
                pair[1] = value
                return

        # Otherwise append a new [key, value] pair to the chain
        bucket.append([key, value])
        self.count += 1

        # Resize if load factor exceeds 0.75
        if self.count / self.size > 0.75:
            self.dynamicResizing()

For inserting the key value pair into the hash tables 

## DELETE method 

    def delete(self,key):
        index = self._hash(key, method="division")
        bucket = self.table[index]

        for i, pair in enumerate(bucket):
            if pair[0] == key:
                bucket.pop(i)
                self.count -= 1
                return pair[1]

        raise KeyError(f"Key '{key}' not found")


Will give an error incase key is not found. 

## DYNAMIC RESIZING

    def dynamicResizing(self):
        old_table = self.table
        # Roughly double the size, pick a prime-ish number
        self.size = self.size * 2 + 1
        self.table = [[] for _ in range(self.size)]
        self.count = 0

        # Re-insert every existing key-value pair
        for bucket in old_table:
            for key, value in bucket:
                self.insert(key, value)

- When the load factor (count / table size) crosses 0.75, the table roughly doubles and every entry is rehashed. 

- This keeps average chain length short — O(1) expected lookup.

### Hash function

    def _hash(self, key, method="division"):
        # Implement division method
        # Implement multiplication method
        # Convert key to an integer regardless of type
        if isinstance(key, int):
            k = abs(key)
        elif isinstance(key, str):
            # Polynomial rolling hash: weighted sum of char codes
            k = 0
            for ch in key:
                k = k * 31 + ord(ch)
        elif isinstance(key, float):
            k = abs(int(key * 1_000_000))
        else:
            k = abs(id(key))

        if method == "division":
            # h(k) = k mod m
            return k % self.size
        elif method == "multiplication":
            # h(k) = floor(m * (k * A mod 1)),  A ≈ (√5 - 1)/2 (Knuth's suggestion)
            A = 0.6180339887498949
            return int(self.size * ((k * A) % 1))
        else:
            raise ValueError(f"Unknown hashing method: {method}")

- For string keys the code uses a polynomial rolling hash (base 31) to spread characters across the integer space before applying the modular reduction. 

- The multiplication method uses Knuth's constant A ≈ (√5 − 1)/2, which produces good distribution because it's maximally irrational.

### LET's TEST IT (while praying that it works..)

In [1]:
from Homework3 import HashMap

In [2]:
hm = HashMap(size=7)

# Basic insert + search
hm.insert("apple", 3)
hm.insert("banana", 5)
hm.insert("cherry", 7)
hm.insert(42, "answer")
print("after inserts :", hm)
print("search apple  :", hm.search("apple")) # should give 3
print("search 42     :", hm.search(42)) # should show answer

# Update existing key
hm.insert("apple", 99) # should update the value
print("updated apple :", hm.search("apple"))

# Delete
hm.delete("banana")
print("after delete  :", hm) #shoult not have apple
print("contains banana:", "banana" in hm) # should be fale

after inserts : {42: 'answer', 'apple': 3, 'banana': 5, 'cherry': 7}
search apple  : 3
search 42     : answer
updated apple : 99
after delete  : {42: 'answer', 'apple': 99, 'cherry': 7}
contains banana: False


### PRAYING WORKED : D